# Group 1 — DDSP Fine-Tuning

Fine-tune a pretrained DDSP Autoencoder on calibration audio to capture a
target instrument's timbre. Freezes the GRU encoder and input FC layers,
training only the output FC, synthesizer parameters, and reverb IR.

**Prerequisites:**
- Run `group1_ddsp_preprocess.ipynb` to download a base model.
- Have a calibration WAV (>= 10s, ideally ~30s chromatic scale).

**Outputs:** A fine-tuned DDSP checkpoint directory.

In [ ]:
import sys
import os
from pathlib import Path

# Add notebooks/ to path so we can import from shared/
sys.path.insert(0, str(Path("..").resolve()))

from shared.ddsp import (
    chunk_for_training,
    finetune,
    preprocess_audio,
)

# --------------- Configuration (edit these) ---------------
BASE_CHECKPOINT_DIR = "../../models/ddsp_pretrained/violin"  # from preprocessing notebook
CALIBRATION_WAV = "../../recordings/calibration.wav"  # >= 10s recording
OUTPUT_DIR = "../../calibrations/finetune_output"

STEPS = 2000
LEARNING_RATE = 0.0001
BATCH_SIZE = 4

In [ ]:
# ---- Preprocess calibration audio ----

features = preprocess_audio(CALIBRATION_WAV)
examples = chunk_for_training(features)

In [ ]:
# ---- Fine-tune ----

finetune(
    base_checkpoint_dir=BASE_CHECKPOINT_DIR,
    examples=examples,
    output_dir=OUTPUT_DIR,
    steps=STEPS,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
)

In [ ]:
# ---- Verify output ----

print(f"Contents of {OUTPUT_DIR}:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    size_kb = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1024
    print(f"  {f:40s} {size_kb:8.1f} KB")